

*   **--------------------------------------------Name: Vanchanagiri Alekhya-----------------------------------------**
*   **----------------------------------------Enrollment Number: MML2025002----------------------------------------**

*   **------------------------------------------GenAI & LLMs Assignment-1--------------------------------------------**


**Use MNIST database of handwritten digits, I think it is a perfect dataset for learning such concepts. Also start solving the problems in such a way using OOM so thatapart form GDA,SGDA, we can implement three optimization algorithms-  ,
RMSProp and ADAM on it and compare the results**


**1. Backpropagation Implementation:**
1.   Uses a fully-connected neural network architecture
1.   Implements forward and backward propagation
1.   Includes mini-batch training
2.   Uses sigmoid activation function
2.   Implements gradient descent optimization ( compare the performance of
all three Batch GDA, SGDA( stochastic gradient descent algorithm),and
mini-batch GDA)

In [28]:
import numpy as np
from keras.datasets import mnist

def one_hot(y, C=10):
    Y = np.zeros((y.size, C))
    Y[np.arange(y.size), y] = 1
    return Y

def accuracy(y_pred, y_true):
    return np.mean(y_pred == y_true)

def relu(Z):
    return np.maximum(0, Z)

def drelu(Z):
    return (Z > 0).astype(float)

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return expZ / np.sum(expZ, axis=1, keepdims=True)

class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers
        self.params = {}
        self.init_params()

    def init_params(self):
        for l in range(1, len(self.layers)):
            self.params[f"W{l}"] = np.random.randn(
                self.layers[l-1], self.layers[l]
            ) * 0.01
            self.params[f"b{l}"] = np.zeros((1, self.layers[l]))

    def forward(self, X):
        self.cache = {"A0": X}
        A = X

        for l in range(1, len(self.layers)-1):
            Z = A @ self.params[f"W{l}"] + self.params[f"b{l}"]
            A = relu(Z)
            self.cache[f"Z{l}"] = Z
            self.cache[f"A{l}"] = A

        ZL = A @ self.params[f"W{len(self.layers)-1}"] + \
             self.params[f"b{len(self.layers)-1}"]
        AL = softmax(ZL)
        self.cache[f"A{len(self.layers)-1}"] = AL
        return AL

    def backward(self, Y):
        grads = {}
        m = Y.shape[0]

        dZ = self.cache[f"A{len(self.layers)-1}"] - Y

        for l in reversed(range(1, len(self.layers))):
            grads[f"dW{l}"] = self.cache[f"A{l-1}"].T @ dZ / m
            grads[f"db{l}"] = np.sum(dZ, axis=0, keepdims=True) / m

            if l > 1:
                dA = dZ @ self.params[f"W{l}"].T
                dZ = dA * drelu(self.cache[f"Z{l-1}"])

        return grads

    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)

class GD:
    def __init__(self, lr):
        self.lr = lr
    def step(self, p, g):
        for k in p:
            p[k] -= self.lr * g["d"+k]

class AdaGrad:
    def __init__(self, lr):
        self.lr = lr
        self.h = {}
    def step(self, p, g):
        for k in p:
            self.h.setdefault(k, np.zeros_like(p[k]))
            self.h[k] += g["d"+k]**2
            p[k] -= self.lr * g["d"+k] / (np.sqrt(self.h[k]) + 1e-8)

class RMSProp:
    def __init__(self, lr):
        self.lr = lr
        self.v = {}
    def step(self, p, g):
        for k in p:
            self.v.setdefault(k, np.zeros_like(p[k]))
            self.v[k] = 0.9*self.v[k] + 0.1*g["d"+k]**2
            p[k] -= self.lr * g["d"+k] / (np.sqrt(self.v[k]) + 1e-8)

class Adam:
    def __init__(self, lr):
        self.lr = lr
        self.m, self.v = {}, {}
        self.t = 0
    def step(self, p, g):
        self.t += 1
        for k in p:
            self.m.setdefault(k, np.zeros_like(p[k]))
            self.v.setdefault(k, np.zeros_like(p[k]))

            self.m[k] = 0.9*self.m[k] + 0.1*g["d"+k]
            self.v[k] = 0.999*self.v[k] + 0.001*(g["d"+k]**2)

            m_hat = self.m[k] / (1 - 0.9**self.t)
            v_hat = self.v[k] / (1 - 0.999**self.t)

            p[k] -= self.lr * m_hat / (np.sqrt(v_hat) + 1e-8)

def train(model, optimizer, X, Y, epochs, batch_size):
    n = X.shape[0]
    for _ in range(epochs):
        idx = np.random.permutation(n)
        X, Y = X[idx], Y[idx]

        for i in range(0, n, batch_size):
            xb = X[i:i+batch_size]
            yb = Y[i:i+batch_size]

            model.forward(xb)
            grads = model.backward(yb)
            optimizer.step(model.params, grads)

if __name__ == "__main__":

    (Xtr, ytr), (Xte, yte) = mnist.load_data()
    Xtr = Xtr.reshape(-1, 784) / 255.0
    Xte = Xte.reshape(-1, 784) / 255.0
    Ytr = one_hot(ytr)

    print("\n===== PART A: Batch vs SGD vs Mini-batch (GD) =====\n")

    gd_modes = {
        "Batch GDA": Xtr.shape[0],
        "SGDA": 1,
        "Mini-batch GDA": 64
    }

    for mode, batch in gd_modes.items():
        model = NeuralNetwork([784, 128, 64, 10])
        optimizer = GD(lr=0.01)

        train(model, optimizer, Xtr, Ytr, epochs=30, batch_size=batch)

        tr_acc = accuracy(model.predict(Xtr), ytr)
        te_acc = accuracy(model.predict(Xte), yte)

        print(f"{mode:15s} | Train:{tr_acc*100:6.2f}% | Test:{te_acc*100:6.2f}%")

    print("\n===== PART B: Optimizer Comparison (Mini-batch=64) =====\n")

    optimizers = {
        "AdaGrad": AdaGrad,
        "RMSProp": RMSProp,
        "Adam": Adam
    }

    for name, opt_class in optimizers.items():
        model = NeuralNetwork([784, 128, 64, 10])
        optimizer = opt_class(lr=0.01)

        train(model, optimizer, Xtr, Ytr, epochs=15, batch_size=64)

        tr_acc = accuracy(model.predict(Xtr), ytr)
        te_acc = accuracy(model.predict(Xte), yte)

        print(f"{name:10s} | Train:{tr_acc*100:6.2f}% | Test:{te_acc*100:6.2f}%")



===== PART A: Batch vs SGD vs Mini-batch (GD) =====

Batch GDA       | Train: 11.24% | Test: 11.35%
SGDA            | Train: 99.78% | Test: 97.94%
Mini-batch GDA  | Train: 96.24% | Test: 95.82%

===== PART B: Optimizer Comparison (Mini-batch=64) =====

AdaGrad    | Train: 97.12% | Test: 96.36%
RMSProp    | Train: 96.72% | Test: 95.38%
Adam       | Train: 98.48% | Test: 96.80%


**2. CNN Implementation:**

1.   Includes convolutional layers, ReLU activation, and max pooling
1.   Has a simple architecture: Conv -> ReLU -> MaxPool -> Conv -> ReLU ->
MaxPool -> FC
1.   Uses MNIST dataset (28x28 grayscale images)
2.   For both implementations use only NumPy for numerical
computations

B.Experimentations to be evaluated after the
theory is covered in the class:


2.   Try different network architectures by changing layer sizes
2.   Experiment with learning rates and batch sizes
1.   Add additional features like:
*   Different activation functions (ReLU, tanh)
*   Dropout regularization
*   Batch normalization
*   Different optimization algorithms (AdaGrad, RMSprop, ADAM)





In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

class CNN(nn.Module):
    def __init__(self, activation="relu", dropout=0.0, batchnorm=False):
        super().__init__()

        act = nn.ReLU() if activation == "relu" else nn.Tanh()

        layers = [
            nn.Conv2d(1, 32, 3),
            nn.BatchNorm2d(32) if batchnorm else nn.Identity(),
            act,
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64) if batchnorm else nn.Identity(),
            act,
            nn.MaxPool2d(2),
        ]

        self.conv = nn.Sequential(*layers)

        self.fc = nn.Sequential(
            nn.Linear(64 * 5 * 5, 128),
            act,
            nn.Dropout(dropout),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

def train(model, loader, optimizer):
    model.train()
    loss_fn = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

def evaluate(model, loader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()

    return correct / len(loader.dataset)

transform = transforms.ToTensor()
train_ds = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST("./data", train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=1000)

EPOCHS = 5
LR = 0.001

print("\n=== Activation Function Comparison ===")

for act in ["relu", "tanh"]:
    model = CNN(activation=act).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for _ in range(EPOCHS):
        train(model, train_loader, optimizer)

    acc = evaluate(model, test_loader)
    print(f"Activation: {act:5s} | Test Accuracy: {acc:.4f}")

print("\n=== Optimizer Comparison ===")

optimizers = {
    "SGD": optim.SGD,
    "AdaGrad": optim.Adagrad,
    "RMSprop": optim.RMSprop,
    "Adam": optim.Adam
}

for name, opt in optimizers.items():
    model = CNN(activation="relu").to(device)
    optimizer = opt(model.parameters(), lr=LR)

    for _ in range(EPOCHS):
        train(model, train_loader, optimizer)

    acc = evaluate(model, test_loader)
    print(f"Optimizer: {name:8s} | Test Accuracy: {acc:.4f}")

print("\n=== Dropout Comparison ===")

for drop in [0.0, 0.5]:
    model = CNN(activation="relu", dropout=drop).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for _ in range(EPOCHS):
        train(model, train_loader, optimizer)

    acc = evaluate(model, test_loader)
    print(f"Dropout: {drop} | Test Accuracy: {acc:.4f}")

print("\n=== Batch Normalization Comparison ===")

for bn in [False, True]:
    model = CNN(activation="relu", batchnorm=bn).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for _ in range(EPOCHS):
        train(model, train_loader, optimizer)

    acc = evaluate(model, test_loader)
    print(f"BatchNorm: {bn} | Test Accuracy: {acc:.4f}")



=== Activation Function Comparison ===
Activation: relu  | Test Accuracy: 0.9888
Activation: tanh  | Test Accuracy: 0.9865

=== Optimizer Comparison ===
Optimizer: SGD      | Test Accuracy: 0.8527
Optimizer: AdaGrad  | Test Accuracy: 0.9584
Optimizer: RMSprop  | Test Accuracy: 0.9906
Optimizer: Adam     | Test Accuracy: 0.9880

=== Dropout Comparison ===
Dropout: 0.0 | Test Accuracy: 0.9905
Dropout: 0.5 | Test Accuracy: 0.9904

=== Batch Normalization Comparison ===
BatchNorm: False | Test Accuracy: 0.9909
BatchNorm: True | Test Accuracy: 0.9902
